*В данном техническом файле Вы можете найти код для вывода необходимых столбцов, а также создания новых на основе предиктивного анализа, с целью вывода единой таблицы для будущего построения дашбордов в программе Tableau.*

**Цель:** создать конечный датасет с необходимыми данными в форме, из которой будет удобно их брать, совершать математические вычисления и предсказывать исход

**Этапы работы**

1. Подключил необходимые библиотеки
2. Загрузил исходный датасет
3. Обучил старые матчи на основе исторических данных с помощью Random Forest
4. На основе предиктивного анализа из предыдущего технического файла вывел прогноз для команд на победу, а также вероятность победы для каждой команды и ничьи.

**Результат:** объединённый csv-файл с 3 ключевыми характеристиками и прогнозами для предсказания исхода матча

## 1. Загружаем исходные матчи

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv('final_dataset.csv')
df['match_date'] = pd.to_datetime(df['match_date'])

## 2. Готовим три признака модели

In [2]:
df['form_difference'] = (df['home_form_points_last5'] - df['away_form_points_last5'])
df['result_number'] = df['match_result'].map({'A': 0, 'D': 1, 'H': 2})

features = [
    'rating_difference',
    'form_difference',
    'home_goals_against_last5'
]

model_history = df.dropna(subset=features + ['result_number']).copy()
X = model_history[features]
y = model_history['result_number'].astype(int)

## 3. Обучаем Random Forest

In [3]:
RF = RandomForestClassifier(n_estimators=100, random_state=42)
RF.fit(X, y)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


## 4. Берём последние показатели каждой команды

In [4]:
home_teams = df[[
    'home_team', 'country_name', 'league_name', 'match_date',
    'home_avg_rating', 'home_form_points_last5',
    'home_goals_against_last5'
]].copy()

home_teams.columns = [
    'team', 'country_name', 'league_name', 'last_match_date',
    'rating', 'form_points_last5', 'goals_against_last5'
]


away_teams = df[[
    'away_team', 'country_name', 'league_name', 'match_date',
    'away_avg_rating', 'away_form_points_last5',
    'away_goals_against_last5'
]].copy()

away_teams.columns = [
    'team', 'country_name', 'league_name', 'last_match_date',
    'rating', 'form_points_last5', 'goals_against_last5'
]

teams = pd.concat([home_teams, away_teams], ignore_index=True)
teams = teams.dropna(subset=['rating', 'form_points_last5', 'goals_against_last5'])
teams = teams.sort_values('last_match_date')
teams = teams.drop_duplicates('team', keep='last').reset_index(drop=True)

all_source_teams = set(pd.concat([df['home_team'], df['away_team']]).dropna())
teams_with_data = set(teams['team'])

## 5. Создаём возможные матчи 2017 года внутри каждой лиги

In [5]:
home = teams.rename(columns={
    'team': 'home_team',
    'rating': 'home_avg_rating',
    'form_points_last5': 'home_form_points_last5',
    'goals_against_last5': 'home_goals_against_last5'
})
[[
    'country_name', 'league_name', 'home_team',
    'home_avg_rating', 'home_form_points_last5',
    'home_goals_against_last5'
]]

away = teams.rename(columns={
    'team': 'away_team',
    'rating': 'away_avg_rating',
    'form_points_last5': 'away_form_points_last5',
    'goals_against_last5': 'away_goals_against_last5'
})
[[
    'country_name', 'league_name', 'away_team',
    'away_avg_rating', 'away_form_points_last5',
    'away_goals_against_last5'
]]

future_matches = home.merge(away, on=['country_name', 'league_name'])
future_matches = future_matches[
    future_matches['home_team'] != future_matches['away_team']
].reset_index(drop=True)

future_matches['rating_difference'] = (
    future_matches['home_avg_rating'] - future_matches['away_avg_rating']
)
future_matches['form_difference'] = (
    future_matches['home_form_points_last5']
    - future_matches['away_form_points_last5']
)

## 6. Считаем результат и вероятности

In [6]:
future_X = future_matches[features]

predicted_numbers = RF.predict(future_X)
future_matches['predicted_result'] = pd.Series(predicted_numbers).map({
    0: 'Победа гостей',
    1: 'Ничья',
    2: 'Победа хозяев'
})

probabilities = RF.predict_proba(future_X)
future_matches['away_win_probability'] = probabilities[:, 0]
future_matches['draw_probability'] = probabilities[:, 1]
future_matches['home_win_probability'] = probabilities[:, 2]

## 7. Оставляем только нужные столбцы и сохраняем единый CSV

In [7]:
result_names = {
    'A': 'Победа гостей',
    'D': 'Ничья',
    'H': 'Победа хозяев'
}

df['match_name'] = df['home_team'] + ' — ' + df['away_team']
df['actual_result'] = df['match_result'].map(result_names)

final_columns = [
    'data_type', 'season', 'match_date', 'country_name',
    'league_name', 'match_name', 'home_team', 'away_team',
    'home_team_goal', 'away_team_goal', 'actual_result',
    'home_avg_rating', 'away_avg_rating', 'rating_difference',
    'home_form_points_last5', 'away_form_points_last5',
    'form_difference', 'home_goals_against_last5',
    'away_goals_against_last5', 'predicted_result',
    'away_win_probability', 'draw_probability',
    'home_win_probability'
]

history_table = df[[
    'season', 'match_date', 'country_name', 'league_name',
    'match_name', 'home_team', 'away_team',
    'home_team_goal', 'away_team_goal', 'actual_result',
    'home_avg_rating', 'away_avg_rating', 'rating_difference',
    'home_form_points_last5', 'away_form_points_last5',
    'form_difference', 'home_goals_against_last5',
    'away_goals_against_last5'
]].copy()
history_table.insert(0, 'data_type', 'Исторический матч')
history_table['predicted_result'] = pd.NA
history_table['away_win_probability'] = pd.NA
history_table['draw_probability'] = pd.NA
history_table['home_win_probability'] = pd.NA
history_table['match_date'] = history_table['match_date'].dt.strftime('%Y-%m-%d')
history_table = history_table[final_columns]

future_table = pd.DataFrame()
future_table['data_type'] = ['Прогноз на 2017'] * len(future_matches)
future_table['season'] = ['2017'] * len(future_matches)
future_table['match_date'] = pd.NA
future_table['country_name'] = future_matches['country_name']
future_table['league_name'] = future_matches['league_name']
future_table['match_name'] = (future_matches['home_team'] + ' — ' + future_matches['away_team'])
future_table['home_team'] = future_matches['home_team']
future_table['away_team'] = future_matches['away_team']
future_table['home_team_goal'] = pd.NA
future_table['away_team_goal'] = pd.NA
future_table['actual_result'] = pd.NA
future_table['home_avg_rating'] = future_matches['home_avg_rating']
future_table['away_avg_rating'] = future_matches['away_avg_rating']
future_table['rating_difference'] = future_matches['rating_difference']
future_table['home_form_points_last5'] = future_matches['home_form_points_last5']
future_table['away_form_points_last5'] = future_matches['away_form_points_last5']
future_table['form_difference'] = future_matches['form_difference']
future_table['home_goals_against_last5'] = future_matches['home_goals_against_last5']
future_table['away_goals_against_last5'] = future_matches['away_goals_against_last5']
future_table['predicted_result'] = future_matches['predicted_result']
future_table['away_win_probability'] = future_matches['away_win_probability']
future_table['draw_probability'] = future_matches['draw_probability']
future_table['home_win_probability'] = future_matches['home_win_probability']
future_table = future_table[final_columns]
future_table = future_table.sort_values(
    ['league_name', 'home_team', 'away_team']
).reset_index(drop=True)

final_table = pd.concat(
    [history_table, future_table],
    ignore_index=True
)

final_table.to_csv(
    'football_dashboard_all_levels_2017.csv',
    index=False,
    encoding='utf-8-sig'
)

C:\Users\dimag\AppData\Local\Temp\ipykernel_23524\3892846231.py:68: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_table = pd.concat(
